In [1]:
from fastai.vision.all import *
import pandas as pd
from fastai.metrics import accuracy_multi, FBetaMulti

metrics = [partial(accuracy_multi, thresh=0.5), FBetaMulti(beta=1)]

# Ensure dataset, df, and test_files are loaded as in your main pipeline
dataset_path = Path("/kaggle/input/wildlife-conservation")

# Define get_x_func
def get_x_func(r):
    return dataset_path/'train_features'/(r['id'] + '.jpg')

# Site-stratified splitter
from sklearn.model_selection import StratifiedGroupKFold
labels_df = pd.read_csv(dataset_path/'train_labels.csv')
site_df = pd.read_csv(dataset_path/'train_features.csv')[['id', 'site']]

df = labels_df.merge(site_df, on='id', how='left').rename(columns={'site': 'site_id'})
label_cols = labels_df.columns[1:]
df['labels'] = df[label_cols].apply(lambda row: ' '.join(label_cols[row.values.astype(bool)]), axis=1)
df['main_label'] = df[label_cols].idxmax(axis=1)

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
X = np.zeros(len(df))
y = df['main_label']
groups = df['site_id']
train_idx, valid_idx = list(sgkf.split(X, y, groups))[0]

def splitter(o):
    return train_idx, valid_idx

# Prepare test files
test_dir = dataset_path / 'test_features'
test_ids = [f.stem for f in test_dir.iterdir() if f.suffix == '.jpg']
test_files = [test_dir / f"{fid}.jpg" for fid in test_ids]

def save_preds_csv(preds, ids, vocab, filename):
    eps = 1e-8
    pred_df = pd.DataFrame(preds.numpy(), columns=vocab)
    pred_df = pred_df.div(pred_df.sum(axis=1) + eps, axis=0)
    pred_df.insert(0, 'id', ids)
    pred_df = pred_df.sort_values('id')
    pred_df.to_csv(filename, index=False)

def make_dls(aug_version='light', img_size=224, bs=32):
    if aug_version == 'light':
        batch_tfms = aug_transforms(flip_vert=True, max_rotate=15, max_zoom=1.1,
                                    max_lighting=0.2, max_warp=0.2, p_affine=0.75, p_lighting=0.75)
    elif aug_version == 'strong':
        batch_tfms = aug_transforms(flip_vert=True, max_rotate=25, max_zoom=1.3,
                                    max_lighting=0.4, max_warp=0.3, p_affine=0.85, p_lighting=0.85) \
                    + [RandomErasing(p=0.7, max_count=2, min_aspect=0.3)]
    elif aug_version == 'mixup':
        batch_tfms = aug_transforms(flip_vert=True, max_rotate=10, max_zoom=1.15,
                                    max_lighting=0.25, max_warp=0.2, p_affine=0.75, p_lighting=0.75)
    else:
        raise ValueError("Unknown aug_version")
    
    dblock = DataBlock(
        blocks=(ImageBlock, MultiCategoryBlock),
        get_x=get_x_func,
        get_y=ColReader('labels', label_delim=' '),
        splitter=splitter,
        item_tfms=Resize(img_size),
        batch_tfms=batch_tfms
    )
    return dblock.dataloaders(df, bs=bs)

In [2]:
# v2_resnet50_strong_cutout.py
dls = make_dls(aug_version='strong', img_size=256, bs=32)

learn = vision_learner(dls, resnet50, loss_func=BCEWithLogitsLossFlat(), metrics=metrics)
learn.fine_tune(5)

test_dl = learn.dls.test_dl(test_files)
preds, _ = learn.tta(dl=test_dl, n=8)

save_preds_csv(preds, test_ids, learn.dls.vocab, 'preds_v2_resnet50_strong_cutout.csv')

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 183MB/s]


epoch,train_loss,valid_loss,accuracy_multi,fbeta_score,time
0,0.363598,0.359132,0.872241,0.092736,02:21


epoch,train_loss,valid_loss,accuracy_multi,fbeta_score,time
0,0.293664,0.319970,0.878604,0.138950,01:51
1,0.246765,0.303355,0.883446,0.303423,01:51
2,0.219024,0.299434,0.887050,0.343871,01:51
3,0.201693,0.284264,0.889696,0.349654,01:52
4,0.198875,0.296298,0.889077,0.352103,01:51
